In [1]:
import networkx as nx

# 1. Creating a custom graph

In [2]:
# Create an empty undirected graph
G = nx.Graph()

In [3]:
C1 = [(0, 1), (1, 2), (2, 3)]
C2 = [(4, 5), (5, 6)]

G.add_edges_from(C1)
G.add_edges_from(C2)
# G.add_edges_from([
#     (1, 2),
#     (2, 3),
#     (3, 4)
# ])

# G.add_edges_from([
#     (5, 6),
#     (6, 7),
# ])

In [4]:
# Verifying components
list(nx.connected_components(G))

[{0, 1, 2, 3}, {4, 5, 6}]

In [5]:
G.nodes

NodeView((0, 1, 2, 3, 4, 5, 6))

In [6]:
len(G.nodes)

7

In [7]:
G.edges

EdgeView([(0, 1), (1, 2), (2, 3), (4, 5), (5, 6)])

In [8]:
len(G.edges)

5

## BFS + Connected components algorithm for PC computing

In [9]:
from collections import deque

def bfs(G: nx.Graph, source: int, visited: set[int]) -> list[int]:
  queue = deque()
  visited.add(source)
  res = []

  # visited[source] = True
  queue.append(source)

  while len(queue) > 0:

    v = queue.popleft()
    res.append(v)
    # print(f"queue: {v}")

    for u in G.neighbors(v):
      if u not in visited:
        visited.add(u)
        queue.append(u)

  return res

In [10]:
def connected_components(G: nx.Graph) -> list[list[int]]:
  # visited = [False] * list(G.nodes)
  visited : set[int] = set()
  components : list[list[int]] = []

  V = list(G.nodes)

  for v in V:
    if v not in visited:
      comp = bfs(G, v, visited)
      components.append(comp)
      # yield comp

  return components

In [11]:
components = connected_components(G)
components

[[0, 1, 2, 3], [4, 5, 6]]

- Considering terminal nodes

In [58]:
def create_custom_graph_with_2_comps(terminal_nodes: list[int]) -> nx.Graph:

  G = nx.Graph()

  C1 = [(1, 2), (2, 3), (3, 4)]
  C2 = [(5, 6), (6, 7)]

  G.add_edges_from(C1)
  G.add_edges_from(C2)

  # Add terminal attribute to nodes
  for v in G.nodes:
    if v in terminal_nodes:
      G.nodes[v]["terminal"] = True
    else:
      G.nodes[v]["terminal"] = False
  
  return G

In [59]:
TERMINAL_NODES = [1, 2, 4, 6, 7]

G = create_custom_graph_with_2_comps(terminal_nodes=TERMINAL_NODES)

In [60]:
G.nodes

NodeView((1, 2, 3, 4, 5, 6, 7))

In [61]:
G.nodes(data="terminal")

NodeDataView({1: True, 2: True, 3: False, 4: True, 5: False, 6: True, 7: True}, data='terminal')

In [62]:
def exclude_non_terminal_nodes_from_component(terminal_nodes: list[int], components: list[list[int]]) -> list[list[int]]:

  excluded_components = []

  for comp in components:
    excluded_comp = []
    for v in comp:
      if v in terminal_nodes:
        excluded_comp.append(v)
      
    excluded_components.append(excluded_comp)

  return excluded_components

In [63]:
components

[[0], [2, 3], [4, 5, 6]]

In [64]:
excluded_comps = exclude_non_terminal_nodes_from_component(TERMINAL_NODES, components)
excluded_comps

[[], [2], [4, 6]]

- Compute Pairwise connectivity considering terminal nodes

In [65]:
import math

def compute_pc_terminals(G: nx.Graph, terminal_nodes: list[int]) -> int:

  # 1. Get connected components
  components = connected_components(G)

  # 2. Exclude non-terminal nodes from the components
  excluded_components = exclude_non_terminal_nodes_from_component(terminal_nodes, components)

  # 3. Compute pairwise connectivity using comb(n, 2) = n! / (n - 2)! * 2!
  pairwise_connectivity = 0

  for comp in excluded_components:
    n = int(len(comp))
    pairwise_connectivity += math.comb(n, 2)

  return pairwise_connectivity

In [66]:
TERMINAL_NODES = [0, 1, 3, 5, 6]

pc = compute_pc_terminals(G, TERMINAL_NODES)
pc

2

In [ ]:
G_copy = G.copy()
H = G_copy.remove_nodes_from(T)

## Compute pairwise connectivity

In [67]:
def compute_pc(G: nx.Graph, S: set, terminal_nodes: list[int]) -> int:

  # Remaining vertices after deletion
  # V_remaining = set(G.nodes) - S
  # num_rem = len(V_remaining)

  T = set(terminal_nodes)

  G_copy = G.copy()
  G_copy.remove_nodes_from(S)

  # print(G_copy.nodes(data="terminal"))

  # 1. Get connected components
  components = connected_components(G_copy)

  print(components)

  # 2. Exclude non-terminal nodes from the components
  excluded_components = []

  for comp in components:
    excluded_comp = []
    for v in comp:
      if v in T:
        excluded_comp.append(v)

    excluded_components.append(excluded_comp) 

  print(f"Excluded components: {excluded_components}")
  
  # 3. Compute pairwise connectivity across terminal nodes over components
  total_pc = 0
  for comp in excluded_components:
    n = len(comp)
    total_pc += n * (n - 1) // 2

  ## More simpler usage:
  # total_pc = 0
  # for comp in connected_components(G_copy):
  #   comp_count = sum(1 for v in comp if v in T)
  #   total_pc += comp_count * (comp_count - 1) // 2

  return total_pc

In [68]:
G.nodes(data="terminal")

NodeDataView({1: True, 2: True, 3: False, 4: True, 5: False, 6: True, 7: True}, data='terminal')

In [69]:
S = set([2, 5])

In [70]:
set(G.nodes) - S

{1, 3, 4, 6, 7}

In [71]:
G.nodes(data="terminal")

NodeDataView({1: True, 2: True, 3: False, 4: True, 5: False, 6: True, 7: True}, data='terminal')

In [72]:
components = connected_components(G)
components

exclude_components = []
for comp in components:
  exclude_comp = []
  print(comp)
  
  for v in comp:
    if v in set(G.nodes) - S and v in TERMINAL_NODES:
      exclude_comp.append(v)

  exclude_components.append(exclude_comp)

exclude_components


[1, 2, 3, 4]
[5, 6, 7]


[[1, 3], [6]]

In [73]:
set(G.nodes) - S

{1, 3, 4, 6, 7}

In [74]:
type(G.nodes - S)

set

In [75]:
G.edges

EdgeView([(1, 2), (2, 3), (3, 4), (5, 6), (6, 7)])

In [76]:
TERMINAL_NODES

[0, 1, 3, 5, 6]

In [77]:
TERMINAL_NODES = [0, 2, 4, 6]

G = create_custom_graph_with_2_comps(terminal_nodes=TERMINAL_NODES)

In [78]:
G.nodes(data="terminal")

NodeDataView({1: False, 2: True, 3: False, 4: True, 5: False, 6: True, 7: False}, data='terminal')

In [79]:
pc = compute_pc(G=G, S={3, 6}, terminal_nodes=TERMINAL_NODES)
pc 

[[1, 2], [4], [5], [7]]
Excluded components: [[2], [4], [], []]


0

## Simple Greedy from ES and from MIS algorithms

In [122]:
def case_1(terminal_nodes, edge_arrangement: 1 | 2 | 3):

  G = nx.Graph()

  if edge_arrangement == 1:
    C = [(1, 2), (1, 4), (2, 5), (2, 3), (3, 6), (4, 5), (5, 6)]

  elif edge_arrangement == 2:
    C = [(1, 2), (1, 3), (2, 4), (2, 5), (3, 4), (4, 6), (5, 6)]

  elif edge_arrangement == 3:
    C = [(1, 2), (1, 5), (2, 3), (2, 4), (3, 6), (4, 6), (5, 4)]

  # C_new = [(edge[0] - 1, edge[1] - 1) for edge in C]

  # print(C_new)

  G.add_edges_from(C)

  for v in G.nodes:
    if v in terminal_nodes:
      G.nodes[v]["terminal"] = True
    else:
      G.nodes[v]["terminal"] = False

  G.nodes(data="terminal")

  print(G.nodes(data="terminal"))

  return G

### Greedy from ES

In [123]:
def greedy_algorithm(
    G: nx.Graph, terminals: list[int], 
    k: int, case: 1 | 2) -> tuple[set, list[int]]:
  
  # If graph is too big, use sparse representation
  # for better memory usage

  T : set = set(terminals)
  V : set = set(G.nodes)

  S : set = set()
  pc_deltas : list[int] = []

  # case 1 : S in V
  # start from the empty set
  # Keep set
  if case == 1:
    K = set()

  # case 2 : S in V \ T (terminal nodes are not deletable)
  # start from the terminal nodes set
  # Keep set 
  if case == 2:
    K = T

  for _ in range(k):

    best_node = None
    best_deg = -1
    best_pc = float("inf")

    # print(f"\nK: iteration\n")
    for node in V - (S | K):
      
      # pc_Sj = None

      S_j = S | {node}
      pc_Sj = compute_pc(G=G, S=S_j, terminal_nodes=terminals)
      print(f"node: {node} and pc Sj : {pc_Sj}")

      # tie-breaker - if two nodes give the same best_pc
      # get the one with higher degree
      # deg = G.degree(node)

      # argmin
      # if pc_Sj < best_pc or (pc_Sj == best_pc and deg > best_deg):
      if pc_Sj <= best_pc:
        best_pc = pc_Sj
        best_node = node
        # best_deg = deg
        print()
        print(f"best node: {best_node} and best gain: {best_pc}")
        print()

    # if best_node is None:
    #   # No feasible solution
    #   break 
    S.add(best_node)
    pc_deltas.append(best_pc)

  return S, pc_deltas

In [124]:
TERMINAL_NODES_1 = [1, 2, 6]

S = greedy_algorithm(G=G, terminals=TERMINAL_NODES_1, k=2, case=2)
S

[[1, 2], [4], [5, 6, 7]]
Excluded components: [[1, 2], [], [6]]
node: 3 and pc Sj : 1

best node: 3 and best gain: 1

[[1, 2, 3], [5, 6, 7]]
Excluded components: [[1, 2], [6]]
node: 4 and pc Sj : 1

best node: 4 and best gain: 1

[[1, 2, 3, 4], [6, 7]]
Excluded components: [[1, 2], [6]]
node: 5 and pc Sj : 1

best node: 5 and best gain: 1

[[1, 2, 3, 4], [5, 6]]
Excluded components: [[1, 2], [6]]
node: 7 and pc Sj : 1

best node: 7 and best gain: 1

[[1, 2], [4], [5, 6]]
Excluded components: [[1, 2], [], [6]]
node: 3 and pc Sj : 1

best node: 3 and best gain: 1

[[1, 2, 3], [5, 6]]
Excluded components: [[1, 2], [6]]
node: 4 and pc Sj : 1

best node: 4 and best gain: 1

[[1, 2, 3, 4], [6]]
Excluded components: [[1, 2], [6]]
node: 5 and pc Sj : 1

best node: 5 and best gain: 1



({5, 7}, [1, 1])

In [152]:
TERMINAL_NODES = [1, 2, 4, 6, 7]
G = create_custom_graph_with_2_comps(TERMINAL_NODES)

print(G.nodes, "\n", G.edges)

[1, 2, 3, 4, 5, 6, 7] 
 [(1, 2), (2, 3), (3, 4), (5, 6), (6, 7)]


In [155]:
H = G.copy()
H.remove_nodes_from(TERMINAL_NODES)

print(H.nodes, "\n", H.edges)

[3, 5] 
 []


In [119]:
S = greedy_algorithm(G=G, terminals=TERMINAL_NODES, k=2, case=1)
S

[[2, 3, 4], [5, 6, 7]]
Excluded components: [[2, 4], [6, 7]]
node: 1 and pc Sj : 2

best node: 1 and best gain: 2

[[1], [3, 4], [5, 6, 7]]
Excluded components: [[1], [4], [6, 7]]
node: 2 and pc Sj : 1

best node: 2 and best gain: 1

[[1, 2], [4], [5, 6, 7]]
Excluded components: [[1, 2], [4], [6, 7]]
node: 3 and pc Sj : 2
[[1, 2, 3], [5, 6, 7]]
Excluded components: [[1, 2], [6, 7]]
node: 4 and pc Sj : 2
[[1, 2, 3, 4], [6, 7]]
Excluded components: [[1, 2, 4], [6, 7]]
node: 5 and pc Sj : 4
[[1, 2, 3, 4], [5], [7]]
Excluded components: [[1, 2, 4], [], [7]]
node: 6 and pc Sj : 3
[[1, 2, 3, 4], [5, 6]]
Excluded components: [[1, 2, 4], [6]]
node: 7 and pc Sj : 3
[[3, 4], [5, 6, 7]]
Excluded components: [[4], [6, 7]]
node: 1 and pc Sj : 1

best node: 1 and best gain: 1

[[1], [4], [5, 6, 7]]
Excluded components: [[1], [4], [6, 7]]
node: 3 and pc Sj : 1

best node: 3 and best gain: 1

[[1], [3], [5, 6, 7]]
Excluded components: [[1], [], [6, 7]]
node: 4 and pc Sj : 1

best node: 4 and best gain

({2, 7}, [1, 0])

In [130]:
TERMINAL_NODES_1 = [1, 2, 6]
G = case_1(TERMINAL_NODES_1, 3)

S = greedy_algorithm(G=G, terminals=TERMINAL_NODES_1, k=2, case=2)
S

[(1, True), (2, True), (5, False), (3, False), (4, False), (6, True)]
[[1, 2, 5, 4, 6]]
Excluded components: [[1, 2, 6]]
node: 3 and pc Sj : 3

best node: 3 and best gain: 3

[[1, 2, 5, 3, 6]]
Excluded components: [[1, 2, 6]]
node: 4 and pc Sj : 3

best node: 4 and best gain: 3

[[1, 2, 3, 4, 6]]
Excluded components: [[1, 2, 6]]
node: 5 and pc Sj : 3

best node: 5 and best gain: 3

[[1, 2, 4, 6]]
Excluded components: [[1, 2, 6]]
node: 3 and pc Sj : 3

best node: 3 and best gain: 3

[[1, 2, 3, 6]]
Excluded components: [[1, 2, 6]]
node: 4 and pc Sj : 3

best node: 4 and best gain: 3



({4, 5}, [3, 3])

In [85]:
G = create_custom_graph_with_2_comps(TERMINAL_NODES)
G

In [86]:
G.nodes

NodeView((1, 2, 3, 4, 5, 6, 7))

In [87]:
G.edges

EdgeView([(1, 2), (2, 3), (3, 4), (5, 6), (6, 7)])

In [88]:
G.remove_node(1)


In [89]:
G.edges

EdgeView([(2, 3), (3, 4), (5, 6), (6, 7)])

In [90]:
G.nodes

NodeView((2, 3, 4, 5, 6, 7))

In [91]:
components =list(nx.connected_components(G))
components

[{2, 3, 4}, {5, 6, 7}]

In [92]:
components = connected_components(G)
components

[[2, 3, 4], [5, 6, 7]]

In [106]:
def greedy_algorithm_new(
  G: nx.Graph, terminals: list[int],
  k: int, case: 1 | 2) -> tuple[set, list[int]]:

  T : set = set(terminals)
  V : set = set(G.nodes)

  S : set = set()
  K : set = set()

  pc_deltas : list[int] = []

  # If number of terminals is equal to budget k
  # then it is the optimal solution

  if len(T) == k:
    non_terminals = V - T
    return non_terminals, [0]
  
  # case 1 : S in V
  # start from the empty set
  # Keep set
  if case == 1:
    K = set()

  # case 2 : S in V \ T (terminal nodes are not deletable)
  # start from the terminal nodes set
  # Keep set 
  if case == 2:
    K = T

  while len(K) != len(V) - k:

    best_node = None
    best_pc = float("inf")

    for j in V - K:

      # compute the PC of the node induced graph of G[V \ K union j]
      S_j = V - (K | {j})
      induced_graph_pc = compute_pc(G=G, S=S_j, terminal_nodes=terminals)
      print(f"node: {j} and pc Sj : {induced_graph_pc}")

      # argmin
      if induced_graph_pc < best_pc:
        best_pc = induced_graph_pc
        best_node = j

    # Update the keep set
    K.add(best_node)
    pc_deltas.append(best_pc)

  S = V - K
  print(f"K: {K} and S: {S}")

  return S, pc_deltas

In [107]:
TERMINAL_NODES = [1, 2, 4, 6, 7]
G = create_custom_graph_with_2_comps(TERMINAL_NODES)

print(G.nodes, "\n", G.edges)

[1, 2, 3, 4, 5, 6, 7] 
 [(1, 2), (2, 3), (3, 4), (5, 6), (6, 7)]


In [120]:
S = greedy_algorithm_new(G=G, terminals=TERMINAL_NODES, k=2, case=1)
S

[[1]]
Excluded components: [[1]]
node: 1 and pc Sj : 0
[[2]]
Excluded components: [[2]]
node: 2 and pc Sj : 0
[[3]]
Excluded components: [[]]
node: 3 and pc Sj : 0
[[4]]
Excluded components: [[4]]
node: 4 and pc Sj : 0
[[5]]
Excluded components: [[]]
node: 5 and pc Sj : 0
[[6]]
Excluded components: [[6]]
node: 6 and pc Sj : 0
[[7]]
Excluded components: [[7]]
node: 7 and pc Sj : 0
[[1, 2]]
Excluded components: [[1, 2]]
node: 2 and pc Sj : 1
[[1], [3]]
Excluded components: [[1], []]
node: 3 and pc Sj : 0
[[1], [4]]
Excluded components: [[1], [4]]
node: 4 and pc Sj : 0
[[1], [5]]
Excluded components: [[1], []]
node: 5 and pc Sj : 0
[[1], [6]]
Excluded components: [[1], [6]]
node: 6 and pc Sj : 0
[[1], [7]]
Excluded components: [[1], [7]]
node: 7 and pc Sj : 0
[[1, 2, 3]]
Excluded components: [[1, 2]]
node: 2 and pc Sj : 1
[[1], [3, 4]]
Excluded components: [[1], [4]]
node: 4 and pc Sj : 0
[[1], [3], [5]]
Excluded components: [[1], [], []]
node: 5 and pc Sj : 0
[[1], [3], [6]]
Excluded com

({2, 7}, [0, 0, 0, 0, 0])

In [134]:
TERMINAL_NODES_1 = [1, 2, 6]
G = case_1(TERMINAL_NODES_1, 3)

S = greedy_algorithm_new(G=G, terminals=TERMINAL_NODES_1, k=2, case=2)
S

[(1, True), (2, True), (5, False), (3, False), (4, False), (6, True)]
[[1, 2, 3, 6]]
Excluded components: [[1, 2, 6]]
node: 3 and pc Sj : 3
[[1, 2, 4, 6]]
Excluded components: [[1, 2, 6]]
node: 4 and pc Sj : 3
[[1, 2, 5], [6]]
Excluded components: [[1, 2], [6]]
node: 5 and pc Sj : 1
K: {1, 2, 5, 6} and S: {3, 4}


({3, 4}, [1])

### Greedy MIS

In [47]:
def compute_pc(G: nx.Graph, S: set, terminal_nodes: list[int]) -> int:

  # Remaining vertices after deletion
  # V_remaining = set(G.nodes) - S
  # num_rem = len(V_remaining)

  T = set(terminal_nodes)

  G_copy = G.copy()
  G_copy.remove_nodes_from(S)

  # print(G_copy.nodes(data="terminal"))

  # 1. Get connected components
  components = connected_components(G_copy)

  print(components)

  # 2. Exclude non-terminal nodes from the components
  excluded_components = []

  for comp in components:
    excluded_comp = []
    for v in comp:
      if v in T:
        excluded_comp.append(v)

    excluded_components.append(excluded_comp) 

  print(f"Excluded components: {excluded_components}")
  
  # 3. Compute pairwise connectivity across terminal nodes over components
  total_pc = 0
  for comp in excluded_components:
    n = len(comp)
    total_pc += n * (n - 1) // 2

  ## More simpler usage:
  # total_pc = 0
  # for comp in connected_components(G_copy):
  #   comp_count = sum(1 for v in comp if v in T)
  #   total_pc += comp_count * (comp_count - 1) // 2

  return total_pc

In [48]:
def greedy_mis_algorithm(
    G: nx.Graph, terminals: list[int], 
    k: int, case: int, mis_trails: int = 3 ) -> tuple[set, list[int]]:
  
  # If graph is too big, use sparse representation
  # for better memory usage
  
  S : set = set()
  V : set = set(G.nodes)
  T : set = set(terminals)

  # Loop for selecting best warm-up MIS
  # which has resulted in lower total PC
  best_pc = float("inf")
  MIS : set = set()

  for _ in range(mis_trails):
    mis = set(nx.maximal_independent_set(G))

    print()
    print(f"mis before T: {mis}")
    print()
    # If case 2, ensure terminal nodes are not deleted
    # keep terminal nodes in MIS
    if case == 2:
      mis = mis | T
    
    # deletions correspond to V \ MIS
    S0 = V - mis

    print()
    print(f"mis after T: {mis}")
    print()
    # feasibility for case 2: do not delete terminal nodes
    if case == 2 and (S0 & T):
      continue

    curr_pc = compute_pc(G=G, S=S0, terminal_nodes=terminals)

    print()
    print(f"PC: {curr_pc}")
    print()
    if curr_pc < best_pc:
      best_pc = curr_pc
      MIS = mis

  print(f"mis found: {MIS}")
  
  pc_deltas : list[int] = []

  while len(MIS) != len(V) - k:

    best_node = None
    best_deg = -1
    best_pc = float("inf")

    for node in V - MIS:

      if case == 1:
        # case 1: S in V
        S_j = V - (MIS | {node})
        pc_Sj = compute_pc(G=G, S=S_j, terminal_nodes=terminals)
        print(f"node: {node} and pc Sj : {pc_Sj}")
        
      if case == 2:
        # case 2: S in V \ T
        if node not in T:
          S_j = V - (MIS | {node})
          pc_Sj = compute_pc(G=G, S=S_j, terminal_nodes=terminals)

      # tie-breaker - if two nodes give the same best_pc
      # get the one with higher degree
      deg = G.degree(node)

      # argmin
      if pc_Sj < best_pc or (pc_Sj == best_pc and deg > best_deg):
        best_pc = pc_Sj
        best_node = node
        best_deg = deg
        # print(f"best node: {best_node} and best gain: {best_pc}")
    
    if best_node is None:
      # No feasible solution
      break

    MIS.add(best_node)
    pc_deltas.append(best_pc)
  
    print(f"mis iter: {MIS}")  
  
  S = V - MIS
  print(f"mis: {MIS} and S: {S}")

  return S, pc_deltas

In [135]:
TERMINAL_NODES = [1, 2, 4, 6, 7]
G = create_custom_graph_with_2_comps(TERMINAL_NODES)

print(G.nodes, "\n", G.edges)

[1, 2, 3, 4, 5, 6, 7] 
 [(1, 2), (2, 3), (3, 4), (5, 6), (6, 7)]


In [137]:
S = greedy_mis_algorithm(G=G, terminals=TERMINAL_NODES, k=2, case=1)
S


mis before T: {2, 4, 5, 7}


mis after T: {2, 4, 5, 7}

[[2], [4], [5], [7]]
Excluded components: [[2], [4], [], [7]]

PC: 0


mis before T: {1, 3, 5, 7}


mis after T: {1, 3, 5, 7}

[[1], [3], [5], [7]]
Excluded components: [[1], [], [], [7]]

PC: 0


mis before T: {1, 3, 5, 7}


mis after T: {1, 3, 5, 7}

[[1], [3], [5], [7]]
Excluded components: [[1], [], [], [7]]

PC: 0

mis found: {2, 4, 5, 7}
[[1, 2], [4], [5], [7]]
Excluded components: [[1, 2], [4], [], [7]]
node: 1 and pc Sj : 1
[[2, 3, 4], [5], [7]]
Excluded components: [[2, 4], [], [7]]
node: 3 and pc Sj : 1
[[2], [4], [5, 6, 7]]
Excluded components: [[2], [4], [6, 7]]
node: 6 and pc Sj : 1
mis iter: {2, 3, 4, 5, 7}
mis: {2, 3, 4, 5, 7} and S: {1, 6}


({1, 6}, [1])

In [142]:
TERMINAL_NODES_1 = [1, 2, 6]
G = case_1(TERMINAL_NODES_1, 3)

S = greedy_mis_algorithm(G=G, terminals=TERMINAL_NODES_1, k=2, case=1)
S

[(1, True), (2, True), (5, False), (3, False), (4, False), (6, True)]

mis before T: {2, 5, 6}


mis after T: {2, 5, 6}

[[2], [5], [6]]
Excluded components: [[2], [], [6]]

PC: 0


mis before T: {1, 3, 4}


mis after T: {1, 3, 4}

[[1], [3], [4]]
Excluded components: [[1], [], []]

PC: 0


mis before T: {1, 3, 4}


mis after T: {1, 3, 4}

[[1], [3], [4]]
Excluded components: [[1], [], []]

PC: 0

mis found: {2, 5, 6}
[[1, 2, 5], [6]]
Excluded components: [[1, 2], [6]]
node: 1 and pc Sj : 1
[[2, 3, 6], [5]]
Excluded components: [[2, 6], []]
node: 3 and pc Sj : 1
[[2, 4, 5, 6]]
Excluded components: [[2, 6]]
node: 4 and pc Sj : 1
mis iter: {2, 4, 5, 6}
mis: {2, 4, 5, 6} and S: {1, 3}


({1, 3}, [1])

In [146]:
def greedy_mis_algorithm_inside_trail(
    G: nx.Graph, terminals: list[int], 
    k: int, case: int, mis_trails: int = 30) -> tuple[set, list[int]]:
  
  # If graph is too big, use sparse representation
  # for better memory usage
  
  S : set = set()
  V : set = set(G.nodes)
  T : set = set(terminals)

  # Loop for selecting best warm-up MIS
  # which has resulted in lower total PC
  best_pc = float("inf")
  MIS : set = set()

  for _ in range(mis_trails):

    mis = set(nx.maximal_independent_set(G))

    # deletions correspond to V \ MIS
    S0 = V - mis

    print()
    print(f"mis after T: {mis}")
    print()
    # feasibility for case 2: do not delete terminal nodes
    if case == 2 and (S0 & T):
      continue

    curr_pc = compute_pc(G=G, S=S0, terminal_nodes=terminals)

    print()
    print(f"PC: {curr_pc}")
    print()

    if curr_pc < best_pc:
      best_pc = curr_pc
      MIS = mis

  # If case 2, ensure terminal nodes are not deleted
  # keep terminal nodes in MIS
  if case == 2:
    mis = mis | T

  print(f"mis found: {MIS}")
  
  pc_deltas : list[int] = []

  while len(MIS) != len(V) - k:

    best_node = None
    best_deg = -1
    best_pc = float("inf")

    for node in V - MIS:

      pc_Sj = None

      if case == 1:
        # case 1: S in V
        S_j = V - (MIS | {node})
        pc_Sj = compute_pc(G=G, S=S_j, terminal_nodes=terminals)
        print(f"node: {node} and pc Sj : {pc_Sj}")
        
      if case == 2:
        # case 2: S in V \ T
        if node not in T:
          S_j = V - (MIS | {node})
          pc_Sj = compute_pc(G=G, S=S_j, terminal_nodes=terminals)

      # tie-breaker - if two nodes give the same best_pc
      # get the one with higher degree
      deg = G.degree(node)

      # argmin
      if pc_Sj is not None and pc_Sj < best_pc or (pc_Sj == best_pc and deg > best_deg):
        best_pc = pc_Sj
        best_node = node
        best_deg = deg
        # print(f"best node: {best_node} and best gain: {best_pc}")
    
    if best_node is None:
      # No feasible solution
      break

    MIS.add(best_node)
    pc_deltas.append(best_pc)
  
    print(f"mis iter: {MIS}")  
  
  S = V - MIS
  print(f"mis: {MIS} and S: {S}")

  return S, pc_deltas

In [151]:
TERMINAL_NODES = [1, 2, 4, 6, 7]
G = create_custom_graph_with_2_comps(TERMINAL_NODES)

print(G.nodes, "\n", G.edges)

S = greedy_mis_algorithm_inside_trail(G=G, terminals=TERMINAL_NODES, k=3, case=2)
S

[1, 2, 3, 4, 5, 6, 7] 
 [(1, 2), (2, 3), (3, 4), (5, 6), (6, 7)]

mis after T: {1, 3, 5, 7}


mis after T: {2, 4, 5, 7}


mis after T: {2, 4, 5, 7}


mis after T: {1, 3, 5, 7}


mis after T: {1, 3, 5, 7}


mis after T: {1, 3, 6}


mis after T: {2, 4, 5, 7}


mis after T: {1, 3, 5, 7}


mis after T: {2, 4, 5, 7}


mis after T: {1, 3, 5, 7}


mis after T: {1, 3, 6}


mis after T: {2, 4, 5, 7}


mis after T: {2, 4, 5, 7}


mis after T: {2, 4, 5, 7}


mis after T: {1, 3, 5, 7}


mis after T: {1, 4, 5, 7}


mis after T: {1, 4, 5, 7}


mis after T: {1, 4, 6}


mis after T: {1, 4, 6}


mis after T: {1, 3, 6}


mis after T: {2, 4, 5, 7}


mis after T: {1, 4, 5, 7}


mis after T: {2, 4, 5, 7}


mis after T: {1, 4, 5, 7}


mis after T: {2, 4, 6}


mis after T: {1, 3, 5, 7}


mis after T: {2, 4, 6}


mis after T: {1, 4, 5, 7}


mis after T: {1, 3, 5, 7}


mis after T: {2, 4, 5, 7}

mis found: set()
[[3]]
Excluded components: [[]]
[[5]]
Excluded components: [[]]
mis iter: {3}
[[3], [5]]
Excluded c

({1, 2, 4, 6, 7}, [0, 0])

In [ ]:
def greedy_mis_algorithm_new_1(
    G: nx.Graph, terminals: list[int], 
    k: int, case: int, mis_trails: int = 20 ) -> tuple[set, list[int]]:
  
  # If graph is too big, use sparse representation
  # for better memory usage
  
  S : set = set()
  V : set = set(G.nodes)
  T : set = set(terminals)
  nonterminals : set = V - T

  # Loop for selecting best warm-up MIS
  # which has resulted in lower total PC
  best_pc = float("inf")
  K : set = set()

  pc_deltas : list[int] = []


  target_keep_count = max(len(V) - k, len(T))
  needed_nonterminals = target_keep_count - len(terminals)

  if needed_nonterminals <= 0:
    kept_vertices = set(terminals)
    deleted_vertices = V - kept_vertices
    return kept_vertices, deleted_vertices

  best_kept_vertices = None
  best_pc = float("inf")

  # Induced graph of nonterminals: H = G[V \ T]
  H = G.copy()
  H.remove_nodes_from(T)
  
  # Kept set
  K : set = None

  for trial_idx in range(mis_trails):
    trial_seed = 42 + trial_idx

    # case 1 : S in V
    # start from the empty set
    # Keep set
    if case == 1:
      K = set()

    # case 2 : S in V \ T (terminal nodes are not deletable)
    # start from the terminal nodes set
    # Keep set 
    if case == 2:
      K = T
        
    # MIS of H
    mis = set(nx.maximal_independent_set(H, seed=trial_seed))

    best_mis_j = None
    best_mis_pc = float("inf")

    for mis_j in mis:
      
      if len(K) != len(V) - k:
        break

      S_j = V - (K | {mis_j})
      induced_graph_pc = compute_pc(G=G, S=S_j, terminal_nodes=terminals)

      # argmin
      if induced_graph_pc < best_mis_pc:
        best_mis_pc = induced_graph_pc
        best_mis_j = mis_j
    
      K.add(best_mis_j)
    # pc_deltas.append(best_mis_pc)
    
    S_mis = V - K
    mis_pc = compute_pc(G=G, S=S_mis, terminal_nodes=terminals)

    # Choosing best kept set
    if mis_pc < best_pc:
      best_kept_vertices = K
      best_pc = mis_pc

  # if |U| < |V| - k:
  if len(best_kept_vertices) < len(V) - k:

    while len(best_kept_vertices) != len(V) - k:
      best_node = None
      best_pc = float("inf")

      for j in V - best_kept_vertices:

        # compute the PC of the node induced graph of G[V \ best_kept_vertices union j]
        S_j = V - (best_kept_vertices | {j})
        induced_graph_pc = compute_pc(G=G, S=S_j, terminal_nodes=terminals)
        print(f"node: {j} and pc Sj : {induced_graph_pc}")

        # argmin
        if induced_graph_pc < best_pc:
          best_pc = induced_graph_pc
          best_node = j

      # Update the keep set
      best_kept_vertices.add(best_node)
      pc_deltas.append(best_pc)

  S = V - best_kept_vertices
  print(f"best_kept_vertices: {best_kept_vertices} and S: {S}")

  return best_kept_vertices, pc_deltas

In [148]:
nx.maximal_independent_set(G)

[5, 3, 7, 1]